# CHiRPE Quickstart with Fused ONNX

This notebook shows the **CHiRPE pipeline** using a fused ONNX model:

1. Load a raw transcript
2. Run CHiRPE preprocessing (`TranscriptPreprocessor`)
3. Score segment summaries with fused ONNX (`string -> logits`)
4. Produce transcript-level prediction with voting

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import onnxruntime as ort
from onnxruntime_extensions import get_library_path

2026-04-23 11:10:31.634229928 [W:onnxruntime:Default, device_discovery.cc:325 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:92 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [2]:
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from repo root or notebooks/.")

sys.path.insert(0, str(REPO_ROOT / "src"))

from chirpe.data.preprocessor import TranscriptPreprocessor


In [3]:
BACKBONE = "bert"  # bert | clinicalbert | mentalbert
BACKBONE_TO_MODEL = {
    "bert": "chirpe_bert_string",
    "clinicalbert": "chirpe_clinicalbert_string",
    "mentalbert": "chirpe_mentalbert_string",
}

if BACKBONE not in BACKBONE_TO_MODEL:
    raise ValueError(f"Unsupported backbone: {BACKBONE}")

MODEL_NAME = BACKBONE_TO_MODEL[BACKBONE]
MODEL_DIR = REPO_ROOT / "outputs" / "string_onnx_fused" / MODEL_NAME
MODEL_PATH = MODEL_DIR / "1" / "model.onnx"
METADATA_PATH = MODEL_DIR / "1" / "export_metadata.json"

CHECKPOINT_DIR = REPO_ROOT / "outputs" / "string_onnx_checkpoints" / BACKBONE / "best_model"
CHIRPE_CONFIG_PATH = CHECKPOINT_DIR / "chirpe_config.json"
DATA_FILE = REPO_ROOT / "data" / "synthetic" / "test.json"

for path in [MODEL_PATH, METADATA_PATH, DATA_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

chirpe_config = {}
if CHIRPE_CONFIG_PATH.exists():
    with open(CHIRPE_CONFIG_PATH, "r") as file:
        chirpe_config = json.load(file)

segmentation_threshold = float(chirpe_config.get("preprocessing", {}).get("segmentation_threshold", 0.8))
max_segments_config = int(chirpe_config.get("preprocessing", {}).get("max_segments_per_transcript", 3))
MAX_SEGMENTS_OVERRIDE = 10
effective_max_segments = MAX_SEGMENTS_OVERRIDE if MAX_SEGMENTS_OVERRIDE is not None else max_segments_config
SAMPLE_INDEX = 2

print(f"Backbone: {BACKBONE}")
print(f"Fused model: {MODEL_NAME}")
print(f"segmentation_threshold={segmentation_threshold}")
print(f"max_segments (config)={max_segments_config}, max_segments (used)={effective_max_segments}")
print(f"sample_index={SAMPLE_INDEX}")

Backbone: bert
Fused model: chirpe_bert_string
segmentation_threshold=0.8
max_segments (config)=2, max_segments (used)=10
sample_index=2


In [4]:
with open(DATA_FILE, "r") as file:
    transcript_items = json.load(file)

if not isinstance(transcript_items, list) or not transcript_items:
    raise RuntimeError("Expected a non-empty list in data file")

if SAMPLE_INDEX < 0 or SAMPLE_INDEX >= len(transcript_items):
    raise IndexError(f"SAMPLE_INDEX out of range: {SAMPLE_INDEX}")
sample_item = transcript_items[SAMPLE_INDEX]
utterances = sample_item.get("transcript", [])
print(f"Loaded {len(transcript_items)} transcript items")
print(f"Using participant: {sample_item.get('participant_id', 'unknown')}")
print(f"Total utterances: {len(utterances)}")
print("\nTranscript preview (first 10 utterances):")
for i, utt in enumerate(utterances[:10], start=1):
    speaker = utt.get("speaker", "unknown")
    text = str(utt.get("text", "")).replace("\n", " " ).strip()
    print(f"{i:02d} | {speaker}: {text}")

Loaded 20 transcript items
Using participant: CHR_0081
Total utterances: 78

Transcript preview (first 10 utterances):
01 | interviewer: Are you superstitious?
02 | interviewee: That describes my experience. I have a strong sense of intuition about things.
03 | interviewer: Do you feel you have a sixth sense?
04 | interviewee: It happens occasionally. I have a strong sense of intuition about things.
05 | interviewer: Do you feel you have a sixth sense?
06 | interviewee: Not particularly
07 | interviewer: Do you believe in omens or signs?
08 | interviewee: I have had that experience. I have a strong sense of intuition about things.
09 | interviewer: Do you feel you have to pay close attention to feel safe?
10 | interviewee: I have felt that way before. I get the sense that I'm being followed sometimes.


In [5]:
preprocessor = TranscriptPreprocessor(
    segmentation_threshold=segmentation_threshold,
    use_llm_summarizer=False,
)

processed = preprocessor.process_transcript(
    sample_item.get("transcript", []),
    participant_id=sample_item.get("participant_id", "unknown"),
)

raw_segments = processed.get("segments", [])[:effective_max_segments]
segments = []
for seg in raw_segments:
    summary = (seg.get("summary") or "").strip()
    if not summary:
        summary = (seg.get("text") or "").strip()
    if summary:
        patched = dict(seg)
        patched["summary"] = summary
        segments.append(patched)

print(f"Raw segments from preprocessor: {processed.get('num_segments', len(processed.get('segments', [])))}")
print(f"Segments kept for inference: {len(segments)}")
for idx, seg in enumerate(segments, start=1):
    preview = seg['summary'][:120].replace('\n', ' ')
    print(f"{idx}. {seg['domain']}: {preview}...")

Raw segments from preprocessor: 15
Segments kept for inference: 10
1. P1_Unusual_Thoughts: Are you superstitious? That describes my experience. I have a strong sense of intuition about things....
2. P13_Superstitious: Do you feel you have a sixth sense? It happens occasionally. I have a strong sense of intuition about things. Do you fee...
3. P2_Suspiciousness: Do you feel you have to pay close attention to feel safe? I have felt that way before. I get the sense that I'm being fo...
4. P12_Derealisation: Does tithem sometimes feel unreal to you? Not particularly Have you felt tithem was moving too fast or slow? Yes, someti...
5. P3_Unusual_Somatic: Have you worried about your body shape? That describes my experience. I've noticed strange sensations in my body lately....
6. P8_Confusion: Do you often feel uncertain about things? It occurs from time to time. I feel uncertain about what's real and what's not...
7. P4_Disoriented: Do you feel puzzled or confused easily? No, not really...
8

In [6]:
with open(METADATA_PATH, "r") as file:
    export_metadata = json.load(file)

ortx_candidates = []
try:
    ortx_candidates.append(Path(get_library_path()))
except Exception:
    pass
if export_metadata.get("ortx_library_required"):
    ortx_candidates.append(Path(export_metadata["ortx_library_required"]))

ortx_library = None
for candidate in ortx_candidates:
    if candidate.exists():
        ortx_library = candidate
        break

if ortx_library is None:
    raise FileNotFoundError(f"ORT Extensions library not found. Tried: {[str(p) for p in ortx_candidates]}")

session_options = ort.SessionOptions()
session_options.register_custom_ops_library(str(ortx_library))
session = ort.InferenceSession(str(MODEL_PATH), session_options, providers=["CPUExecutionProvider"])

print("Inputs:", [{"name": i.name, "type": i.type, "shape": i.shape} for i in session.get_inputs()])
print("Outputs:", [{"name": o.name, "type": o.type, "shape": o.shape} for o in session.get_outputs()])

Inputs: [{'name': 'text', 'type': 'tensor(string)', 'shape': [None]}]
Outputs: [{'name': 'logits', 'type': 'tensor(float)', 'shape': ['batch_size', 2]}]


In [7]:
label_map = {0: "Healthy", 1: "CHR-P"}
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

def softmax(logits: np.ndarray) -> np.ndarray:
    shifted = logits - logits.max(axis=-1, keepdims=True)
    exp_vals = np.exp(shifted)
    return exp_vals / exp_vals.sum(axis=-1, keepdims=True)

def infer_summary(summary_text: str) -> dict:
    # Fused tokenizer path is used as one-string inference per call.
    logits = session.run([output_name], {input_name: np.array([summary_text], dtype=object)})[0]
    probs = softmax(logits)
    pred = int(np.argmax(probs, axis=-1)[0])
    return {
        "prediction": pred,
        "label": label_map.get(pred, str(pred)),
        "confidence": float(probs[0, pred]),
        "logits": logits[0].tolist(),
        "probabilities": probs[0].tolist(),
    }

def run_chirpe_fused_quickstart(transcript_item: dict) -> dict:
    processed_item = preprocessor.process_transcript(
        transcript_item.get("transcript", []),
        participant_id=transcript_item.get("participant_id", "unknown"),
    )

    raw_segs = processed_item.get("segments", [])[:effective_max_segments]
    segs = []
    for s in raw_segs:
        summary = (s.get("summary") or "").strip()
        if not summary:
            summary = (s.get("text") or "").strip()
        if summary:
            patched = dict(s)
            patched["summary"] = summary
            segs.append(patched)
    if not segs:
        raise RuntimeError("No usable segment summaries found for inference")

    segment_results = []
    for seg in segs:
        pred = infer_summary(seg["summary"])
        segment_results.append({
            "domain": seg["domain"],
            "summary": seg["summary"],
            "prediction": pred["prediction"],
            "label": pred["label"],
            "confidence": pred["confidence"],
            "probabilities": pred["probabilities"],
        })

    seg_preds = np.array([row["prediction"] for row in segment_results], dtype=np.int64)
    seg_probs = np.array([row["probabilities"] for row in segment_results], dtype=np.float32)

    majority_pred = int(np.bincount(seg_preds).argmax())
    average_pred = int(np.argmax(seg_probs.mean(axis=0)))

    return {
        "participant_id": processed_item.get("participant_id"),
        "num_segments": len(segment_results),
        "majority_voting": {
            "prediction": majority_pred,
            "label": label_map.get(majority_pred, str(majority_pred)),
        },
        "average_voting": {
            "prediction": average_pred,
            "label": label_map.get(average_pred, str(average_pred)),
            "mean_probabilities": seg_probs.mean(axis=0).tolist(),
        },
        "segment_predictions": segment_results,
    }

In [8]:
result = run_chirpe_fused_quickstart(sample_item)
print(json.dumps(result, indent=2))

{
  "participant_id": "CHR_0081",
  "num_segments": 10,
  "majority_voting": {
    "prediction": 0,
    "label": "Healthy"
  },
  "average_voting": {
    "prediction": 1,
    "label": "CHR-P",
    "mean_probabilities": [
      0.46806374192237854,
      0.5319362878799438
    ]
  },
  "segment_predictions": [
    {
      "domain": "P1_Unusual_Thoughts",
      "summary": "Are you superstitious? That describes my experience. I have a strong sense of intuition about things.",
      "prediction": 1,
      "label": "CHR-P",
      "confidence": 0.5155178904533386,
      "probabilities": [
        0.4844821095466614,
        0.5155178904533386
      ]
    },
    {
      "domain": "P13_Superstitious",
      "summary": "Do you feel you have a sixth sense? It happens occasionally. I have a strong sense of intuition about things. Do you feel you have a sixth sense? Not particularly Do you believe in omens or signs? I have had that experience. I have a strong sense of intuition about things.",
   

This is the CHiRPE quickstart flow with fused ONNX as the inference backend.